In [ ]:
from dolfinx import log, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import pyvista
import numpy as np
import ufl

from mpi4py import MPI
from dolfinx import fem, mesh, plot

from pathlib import Path
from dolfinx import io

import uniaxialGeometry

domain, facet_tag = uniaxialGeometry.create_uniaxial_geometry(h=4)

In [ ]:
# import pyvista

# cells, types, x = plot.vtk_mesh(domain)
# grid = pyvista.UnstructuredGrid(cells, types, x)
# plotter = pyvista.Plotter()
# plotter.add_mesh(grid, show_edges=True)
# plotter.show()

# Weak form

In [ ]:
import basix.ufl
el_u = basix.ufl.element("Lagrange", domain.basix_cell(), 2, shape=(domain.geometry.dim,))
el_p = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
el_mixed = basix.ufl.mixed_element([el_u, el_p])
W = fem.functionspace(domain, el_mixed)

w = fem.Function(W)

In [ ]:

V_u = W.sub(0) # displacement subspace 
V_uCollapsed, V_uCollapsed_to_Vu = V_u.collapse()

u_D_left = fem.Function(V_uCollapsed)
left_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tag.find(1000))

u_D_right = fem.Function(V_uCollapsed)
right_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tag.find(2000))

bcs = [fem.dirichletbc(u_D_left, left_dofs, V_u), fem.dirichletbc(u_D_right, right_dofs, V_u)]

In [ ]:
B = fem.Constant(domain, default_scalar_type((0, 0, 0)))
T = fem.Constant(domain, default_scalar_type((0, 0, 0)))

In [ ]:

# v = ufl.TestFunction(V)
# u = fem.Function(V)

# u, p = ufl.TrialFunctions(W)
(u, p) = ufl.split(w)
v_u, v_p = ufl.TestFunctions(W)


In [ ]:
# Spatial dimension
d = len(u)

# Identity tensor
I = ufl.variable(ufl.Identity(d))

# Deformation gradient
F = ufl.variable(I + ufl.grad(u))

# Right Cauchy-Green tensor
C = ufl.variable(F.T * F)

# Invariants of deformation tensors
I_1 = ufl.variable(ufl.tr(C))
J = ufl.variable(ufl.det(F))
I_1_bar = ufl.variable(J**(-2/3) * I_1) 

In [ ]:
E = default_scalar_type(1.0e4)
nu = default_scalar_type(0.3)
mu = fem.Constant(domain, E / (2 * (1 + nu)))
lmbda = fem.Constant(domain, E * nu / ((1 + nu) * (1 - 2 * nu)))

# psi = (mu / 2) * (I_1 - 3) - mu * ufl.ln(J) + (lmbda / 2) * (ufl.ln(J)) ** 2 # compressible 
psi = (mu / 2) * (I_1_bar - 3) + p*(J-1) # compressible , mixed with p = -kappa (J-1)
# psi = (mu / 2) * (I_1 - 3) # incompressible
P = ufl.diff(psi, F)

# To illustrate the difference between linear and hyperelasticity, the following lines can be uncommented to solve the linear elasticity problem.
# P = 2.0 * mu * ufl.sym(ufl.grad(u)) + lmbda * ufl.tr(ufl.sym(ufl.grad(u))) * I

Define the variational form with traction integral over all facets with value 2.
We set the quadrature degree for the integrals to 4.

In [ ]:
metadata = {"quadrature_degree": 4}
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tag, metadata=metadata)
dx = ufl.Measure("dx", domain=domain, metadata=metadata)

In [ ]:
# residual = (
#     ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((p+1), v_p)*dx + ufl.inner((J-1), v_p)*dx
# )

residual = (
    ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((J-1), v_p)*dx
)

# residual = (
#     ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner(p*(J-1), v_p) * dx
# )

# residualVascularBiomechanics = (
#     ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((J-1), v_p) * dx
# ) # ????????

In [ ]:
pass

# Solving

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_monitor": None,
    "snes_atol": 1e-8,
    "snes_rtol": 1e-8,
    "snes_stol": 1e-8,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
problem = NonlinearProblem(
    residual,
    w,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="hyperelasticity",
)

In [ ]:
log.set_log_level(log.LogLevel.INFO)
displacementPerTimeStep = 15 # uniaxial specimen in 150 long (including )


folder = Path("results")
folder.mkdir(exist_ok=True, parents=True)
xdmf = io.XDMFFile(MPI.COMM_WORLD, folder/"NiklasTesting2.xdmf", "w")
xdmf.write_mesh(domain)
# xdmf.write_meshtags(facet_tags, domain.geometry)


for t in range(0, 11):
    print(f"Time step {t}")
    u_D_right.x.array[0::3] = t*displacementPerTimeStep
    problem.solve()
    converged = problem.solver.getConvergedReason()
    num_its = problem.solver.getIterationNumber()
    print(f"Solver convergence: {converged}. Number of iterations {num_its}, Load {T.value}")
    assert converged > 0, f"Solver did not converge. "


    F_post = ufl.variable(I + ufl.grad(w.sub(0).collapse()))
    C_post = ufl.variable(F_post.T * F_post)
    I_1_post = ufl.variable(ufl.tr(C_post))
    J_post = ufl.variable(ufl.det(F_post))
    I_1_bar_post = ufl.variable(J_post**(-2/3) * I_1_post)
    psi_post = ufl.variable((mu / 2) * (I_1_bar_post - 3) + w.sub(1).collapse()*(J_post-1))
    P_post = ufl.diff(psi_post, F_post)
    sigma_post = ufl.variable(P_post*J_post*ufl.inv(F_post.T))

    # u 
    V_u_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,)))
    u_out = fem.Function(V_u_out)
    u_out.name = "u"
    u_out.interpolate(w.sub(0).collapse())
    xdmf.write_function(u_out, t)

    # p
    V_p_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1))
    p_out = fem.Function(V_p_out)
    p_out.name = "p"
    p_out.interpolate(w.sub(1).collapse())
    xdmf.write_function(p_out, t)

    # J
    V_J_post = fem.functionspace(domain, ("Lagrange", 1))
    J_post = fem.Expression(J_post, V_J_post.element.interpolation_points)
    J_out = fem.Function(V_J_post)
    J_out.name = "J"
    J_out.interpolate(J_post)
    xdmf.write_function(J_out, t)

    # J non-post
    V_J_post2 = fem.functionspace(domain, ("Lagrange", 1))
    J_post2 = fem.Expression(J, V_J_post2.element.interpolation_points)
    J_out2 = fem.Function(V_J_post2)
    J_out2.name = "J_non-post"
    J_out2.interpolate(J_post2)
    xdmf.write_function(J_out2, t)

    # P 
    V_P_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    P_post = fem.Expression(P_post, V_P_post.element.interpolation_points)
    P_out = fem.Function(V_P_post)
    P_out.name = "P"
    P_out.interpolate(P_post)
    xdmf.write_function(P_out, t)

    # P2 non-post
    V_P_post2 = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    P_post2 = fem.Expression(P, V_P_post2.element.interpolation_points)
    P_out2 = fem.Function(V_P_post2)
    P_out2.name = "P_non-post"
    P_out2.interpolate(P_post2)
    xdmf.write_function(P_out2, t)

    # sigma 
    V_sigma_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    sigma_post = fem.Expression(sigma_post, V_sigma_post.element.interpolation_points)
    sigma_out = fem.Function(V_sigma_post)
    sigma_out.name = "sigma"
    sigma_out.interpolate(sigma_post)
    xdmf.write_function(sigma_out, t)


xdmf.close()